# Chargement des données et des modules

In [1]:
# Chargement des librairies
import pandas as pd
import numpy as np

# Affichage propre des graphiques
%matplotlib inline

In [2]:
# Chargement des données
disasters = pd.read_csv("data/disasters_1900_2021.csv")
conflicts = pd.read_csv("data/conflicts.csv")

In [3]:
# Apercu du jeu de donnée Natural Disasters
disasters.head(5)

,Year,Seq,Glide,Disaster Group,Disaster Subgroup,Disaster Type,Disaster Subtype,Disaster Subsubtype,Event Name,Country,...,No Affected,No Homeless,Total Affected,Insured Damages ('000 US$),Total Damages ('000 US$),CPI,Adm Level,Admin1 Code,Admin2 Code,Geo Locations
0,1900,9002,NaN,Natural,Climatological,Drought,Drought,NaN,NaN,Cabo Verde,...,NaN,NaN,NaN,NaN,NaN,3.221647,NaN,NaN,NaN,NaN
1,1900,9001,NaN,Natural,Climatological,Drought,Drought,NaN,NaN,India,...,NaN,NaN,NaN,NaN,NaN,3.221647,NaN,NaN,NaN,NaN
2,1902,12,NaN,Natural,Geophysical,Earthquake,Ground movement,NaN,NaN,Guatemala,...,NaN,NaN,NaN,NaN,25000.0,3.350513,NaN,NaN,NaN,NaN
3,1902,3,NaN,Natural,Geophysical,Volcanic activity,Ash fall,NaN,Santa Maria,Guatemala,...,NaN,NaN,NaN,NaN,NaN,3.350513,NaN,NaN,NaN,NaN
4,1902,10,NaN,Natural,Geophysical,Volcanic activity,Ash fall,NaN,Santa Maria,Guatemala,...,NaN,NaN,NaN,NaN,NaN,3.350513,NaN,NaN,NaN,NaN


In [4]:
# Apercu du jeu de donnée Conflicts
conflicts.head(5)

,conflict_id,location,side_a,side_a_id,side_a_2nd,side_b,side_b_id,side_b_2nd,incompatibility,territory_name,...,ep_end,ep_end_date,ep_end_prec,gwno_a,gwno_a_2nd,gwno_b,gwno_b_2nd,gwno_loc,region,version
0,11342,India,Government of India,141,NaN,GNLA,1163,NaN,1,Garoland,...,1,2012-12-21,NaN,750,NaN,NaN,NaN,750,3,25.1
1,11342,India,Government of India,141,NaN,GNLA,1163,NaN,1,Garoland,...,1,2014-11-27,NaN,750,NaN,NaN,NaN,750,3,25.1
2,11343,"Egypt, Israel",Government of Egypt,117,NaN,Government of Israel,121,NaN,1,Suez/Sinai,...,1,1967-06-10,NaN,651,NaN,666,NaN,"651, 666",2,25.1
3,11343,"Egypt, Israel",Government of Egypt,117,NaN,Government of Israel,121,NaN,1,Suez/Sinai,...,0,NaN,NaN,651,NaN,666,NaN,"651, 666",2,25.1
4,11343,"Egypt, Israel",Government of Egypt,117,NaN,Government of Israel,121,NaN,1,Suez/Sinai,...,1,1970-08-07,NaN,651,NaN,666,NaN,"651, 666",2,25.1


# Nettoyage des données 

## Pour le jeu de donnée "Natural Disasters"

### Ajout des filtres pour les intervalles de dates

### Sélection des colonnes utiles

In [5]:
disasters = disasters[['Year', 'Country', 'Disaster Type', 'Total Deaths', 'Total Affected']]

In [6]:
disasters.head(5)

,Year,Country,Disaster Type,Total Deaths,Total Affected
0,1900,Cabo Verde,Drought,11000.0,NaN
1,1900,India,Drought,1250000.0,NaN
2,1902,Guatemala,Earthquake,2000.0,NaN
3,1902,Guatemala,Volcanic activity,1000.0,NaN
4,1902,Guatemala,Volcanic activity,6000.0,NaN


### Renommage des colonnes

In [7]:
# Afin d'enlever les espaces 
disasters = disasters.rename(columns={
    'Year': 'Year',
    'Country': 'Country',
    'Disaster Type': 'Disaster_type',
    'Total Deaths': 'Total_deaths',
    'Total Affected': 'Total_affected'
})

### Filtrage de 1950-2021

In [8]:
disasters = disasters[disasters['Year'].between(1950,2021)]

### Gestion des valeurs manquantes

La valeur manquante dans le jeu de donnée peut autant signifier qu'il n'y a **pas eu de morts** dans le cas de la colonne **Total Deaths** ou que la donnée soit vraiment **manquante**.

In [9]:
# Regardons le nombre de valeurs manquantes de la colonne Total Deaths par type de catastrophe naturelle
print(disasters.groupby('Disaster_type')['Total_deaths'].apply(
    lambda x: x.isnull().sum()
).sort_values(ascending=False))

Disaster_type
Flood                    1507
Storm                    1156
Drought                   679
Earthquake                366
Wildfire                  270
Epidemic                  228
Volcanic activity         171
Extreme temperature       113
Insect infestation         93
Landslide                  35
Mass movement (dry)         2
Impact                      1
Animal accident             0
Fog                         0
Glacial lake outburst       0
Name: Total_deaths, dtype: int64


On remarque que les valeurs manquantes sont concentrées sur les 3 catastrophes naturelles: **inondation, tempête, sécheresse**. Ce qui est logique étant donné que ce sont les catastrophes naturelles **les moins mortelles** pour la population. Donc, les valeurs manquantes ici représentent réellement une donnée. Pour les comptabiliser, on va les remplacer par 0.

In [10]:
# Remplir les valeurs manquantes par 0
disasters['Total_deaths'] = disasters['Total_deaths'].fillna(0)
disasters['Total_affected'] = disasters['Total_affected'].fillna(0)

# Imputation par la médiane par type de catastrophe
columns = ['Total_deaths', 'Total_affected']

for col in columns:
    disasters[col] = disasters.groupby('Country')[col].transform(
        lambda x: x.fillna(x.median())
    )
    disasters[col] = disasters[col].fillna(disasters[col].median())

In [13]:
# Aggrégation par année et par pays
disasters_agg = disasters.groupby(['Country', 'Year']).agg(
    nb_disasters     = ('Disaster_type', 'count'),
    total_deaths_per_country = ('Total_deaths', 'sum'),
    total_affected   = ('Total_affected', 'sum')
).reset_index()

In [14]:
disasters_agg.head(5)

,Country,Year,nb_disasters,total_deaths_per_country,total_affected
0,Afghanistan,1954,1,2000.0,0.0
1,Afghanistan,1956,2,151.0,2000.0
2,Afghanistan,1963,1,107.0,0.0
3,Afghanistan,1969,1,0.0,48000.0
4,Afghanistan,1971,2,100.0,0.0


## Pour le jeux de donnée "Conflicts"

### Renommage des colonnes 

1 = Conflit mineur : entre 25 et 999 morts au combat au cours d'une année donnée. 2 = Guerre : au moins 1 000 morts au combat au cours d'une année donnée. 

1 = extrasystémique (entre un État et un groupe non étatique situé hors de son propre territoire, où la partie gouvernementale lutte pour conserver le contrôle d'un territoire situé en dehors du système étatique) 2 = interétatique (les deux parties sont des États dans le système de classification de Gleditsch et Ward) 3 = intra-étatique (la partie A est toujours un gouvernement ; la partie B est toujours un ou plusieurs groupes rebelles ; il n'y a pas d'implication de gouvernements étrangers avec des troupes, c'est-à-dire qu'il n'y a pas de code side_a_2nd ou side_b_2nd) 4 = intra-étatique internationalisé (le camp A est toujours un gouvernement ; le camp B est toujours un ou plusieurs groupes rebelles ; il y a implication de gouvernements étrangers avec des troupes, c'est-à-dire qu'il y a au moins UN code side_a_2nd ou side_b_2nd) 



In [15]:
conflicts = conflicts.rename(columns={
    'year' : 'year',
    'location': 'Country'
})

### Filtrage pour l'année 1950 jusqu'à 2021

In [16]:
conflicts =conflicts[conflicts['year'].between(1950, 2021)]

### Sélection des colonnes essentielles

In [18]:
conflicts =conflicts[['Country', 'year', 'intensity_level', 'type_of_conflict']]

In [20]:
conflicts.head(5)

,Country,year,intensity_level,type_of_conflict
0,India,2012,1,3
1,India,2014,1,3
2,"Egypt, Israel",1967,2,2
3,"Egypt, Israel",1969,1,2
4,"Egypt, Israel",1970,1,2
